# LeetCode 1093 — Statistics from a Large Sample bằng Python

Link đề: https://leetcode.com/problems/statistics-from-a-large-sample/ · Độ khó: **Medium** · Chủ đề: Array, Math, Probability and Statistics

Skeleton LeetCode:

```python
class Solution:
    def sampleStats(self, count: List[int]) -> List[float]:
        ...
```

Ghi chú: kết quả là **mảng `5` số thực** `[minimum, maximum, mean, median, mode]` và đề chấp nhận sai số `10^-5`, nên notebook so sánh bằng hàm `close` (sai số tuyệt đối `≤ 10^-5` từng phần tử), không so `==` trên số thực. Các điểm cần đọc kỹ: **trung vị** của mẫu có số phần tử **chẵn** là trung bình hai phần tử giữa; **mode** được đảm bảo **duy nhất**; tổng số phần tử tới `10^9` nên **không được** dựng lại mẫu. Điểm cốt lõi: chỉ có `256` giá trị nên mọi thống kê tính được bằng **quét một mảng `256` ô** với số học trên số lượng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mẫu số nguyên trong `[0, 255]` được nén bằng mảng đếm `count[k]`; tính `min`, `max`, `mean`, `median`, `mode`.
2. Nắm công cụ: tổng có trọng số, tổng tích luỹ (prefix sum), tìm phần tử thứ `k` theo thứ tự sắp xếp, `bisect`, số học chính xác với `Fraction`.
3. Đi từ **dựng lại mẫu rồi tính** → **quét `256` ô + tìm phần tử thứ `k`** → **hai con trỏ từ hai đầu** → **tổng tích luỹ + `bisect`** → **số học chính xác bằng `Fraction`**.
4. Hiểu mấu chốt: không cần danh sách `10^9` phần tử; trung vị là trung bình của phần tử thứ `(n+1)//2` và thứ `n//2 + 1` (cùng chỉ số khi `n` lẻ), tìm bằng tổng tích luỹ của `count`.
5. Tránh lỗi kinh điển: chia nguyên khi tính trung bình, quên trường hợp `n` chẵn, lấy `min/max` bằng chỉ số `0`/`255` thay vì giá trị có mặt thật, tính trung bình không có trọng số, chọn `mode` theo giá trị thay vì theo số lần.

## 2. Đề bài & ví dụ

Cho một mẫu lớn các số nguyên trong khoảng `[0, 255]`. Vì mẫu rất lớn nên nó được biểu diễn bằng mảng `count`, trong đó `count[k]` là **số lần** `k` xuất hiện trong mẫu.

Hãy tính:

- `minimum`: phần tử nhỏ nhất;
- `maximum`: phần tử lớn nhất;
- `mean`: tổng mọi phần tử chia cho số phần tử;
- `median`: nếu số phần tử **lẻ** là phần tử giữa sau khi sắp xếp; nếu **chẵn** là trung bình của **hai** phần tử giữa;
- `mode`: số xuất hiện nhiều nhất (**đảm bảo duy nhất**).

Trả về `[minimum, maximum, mean, median, mode]` dưới dạng số thực; sai số `10^-5` được chấp nhận.

**Ví dụ 1:** mẫu `[1,2,2,2,3,3,3,3]` (`count[1]=1, count[2]=3, count[3]=4`) → `[1, 3, 2.375, 2.5, 3]`.

**Ví dụ 2:** mẫu `[1,1,1,1,2,2,2,3,3,4,4]` (`count[1]=4, count[2]=3, count[3]=2, count[4]=2`) → `[1, 4, 2.18182, 2, 1]`.

**Ràng buộc:** `count.length == 256`; `0 <= count[i] <= 10^9`; `1 <= sum(count) <= 10^9`; mode duy nhất.

In [1]:
import itertools
import random
import statistics
import time
from bisect import bisect_left
from fractions import Fraction
from itertools import accumulate
from typing import List


def from_sample(values):
    """Dựng mảng đếm 256 ô từ một danh sách số nguyên (dùng cho test và bài tập)."""
    c = [0] * 256
    for v in values:
        c[v] += 1
    return c


def close(a, b, tol=1e-5):
    return len(a) == len(b) and all(abs(x - y) <= tol for x, y in zip(a, b))


examples = [
    (from_sample([1, 2, 2, 2, 3, 3, 3, 3]), [1, 3, 2.375, 2.5, 3]),
    (from_sample([1, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4]), [1, 4, 2.18182, 2, 1]),
]
for c, want in examples:
    print({k: v for k, v in enumerate(c) if v}, "->", want)

{1: 1, 2: 3, 3: 4} -> [1, 3, 2.375, 2.5, 3]
{1: 4, 2: 3, 3: 2, 4: 2} -> [1, 4, 2.18182, 2, 1]


## 3. Kiến thức nền

### 3.1 Thống kê từ bảng đếm

Với mảng đếm `count` và `n = Σ count[k]`:

- tổng `S = Σ k · count[k]`, `mean = S / n`;
- `min` = giá trị `k` nhỏ nhất có `count[k] > 0`, `max` = giá trị lớn nhất có `count[k] > 0`;
- `mode` = `k` có `count[k]` lớn nhất (duy nhất nên không cần phá hoà).

### 3.2 Phần tử thứ `k` trong thứ tự sắp xếp

Nếu viết mẫu đã sắp xếp `x_1 ≤ x_2 ≤ … ≤ x_n` thì `x_j = v` với `v` là giá trị nhỏ nhất có tổng tích luỹ `count[0] + … + count[v] ≥ j`. Ta tìm được `x_j` bằng một lần quét `256` ô hoặc bằng `bisect` trên mảng tổng tích luỹ.

In [2]:
def kth_smallest(count, j):
    """x_j (1-based) của mẫu được biểu diễn bởi count, bằng một lần quét tích luỹ."""
    run = 0
    for v in range(256):
        run += count[v]
        if run >= j:
            return v
    raise ValueError("j vượt quá kích thước mẫu")


c = examples[0][0]
print([kth_smallest(c, j) for j in range(1, 9)])
assert [kth_smallest(c, j) for j in range(1, 9)] == [1, 2, 2, 2, 3, 3, 3, 3]

[1, 2, 2, 2, 3, 3, 3, 3]


### 3.3 Trung vị bằng hai phần tử giữa

Với `n` phần tử: `n` lẻ thì phần tử giữa là `x_{(n+1)/2}`; `n` chẵn thì là trung bình của `x_{n/2}` và `x_{n/2+1}`. Cả hai trường hợp gộp thành

`median = (x_{(n+1)//2} + x_{n//2 + 1}) / 2`

vì khi `n` lẻ thì `(n+1)//2 = n//2 + 1` nên hai số hạng bằng nhau.

### 3.4 Sai số số thực

Đề chấp nhận sai số `10^-5`; ta dùng `close(a, b)`. Với `n` tới `10^9` và `S` tới `255 · 10^9` (cỡ `2.55·10^11`), phép chia số nguyên Python `S / n` đúng (làm tròn chính xác) nên không lo mất độ chính xác.

## 4. Ý tưởng nền tảng / trực giác

Mẫu có thể có `10^9` phần tử nhưng chỉ có **`256` giá trị khác nhau**. Mọi đại lượng đều là hàm của bảng đếm: `min/max` là biên của phần khác `0`; `mean` là trung bình có trọng số; `mode` là ô đếm lớn nhất; `median` nằm ở ô nơi tổng tích luỹ **vượt qua** `n/2`. Một lần quét `256` ô là đủ — thời gian hằng số theo kích thước mẫu.

## 5. Brute force — dựng lại mẫu rồi tính theo định nghĩa

**Ý tưởng:** dựng danh sách `sample` (lặp `k` đúng `count[k]` lần, đã sắp xếp), rồi tính `min`, `max`, `mean`, `median`, `mode` bằng định nghĩa. Đúng nhưng cần `O(n)` bộ nhớ; chỉ dùng cho mẫu nhỏ làm **oracle**.

In [3]:
def stats_brute(count):
    sample = [k for k in range(256) for _ in range(count[k])]          # đã sắp xếp
    n = len(sample)
    mean = sum(sample) / n
    if n % 2:
        median = sample[n // 2]
    else:
        median = (sample[n // 2 - 1] + sample[n // 2]) / 2
    freq = {}
    for v in sample:
        freq[v] = freq.get(v, 0) + 1
    mode = max(freq, key=freq.get)
    return [float(sample[0]), float(sample[-1]), mean, float(median), float(mode)]


for c, want in examples:
    assert close(stats_brute(c), want), stats_brute(c)
print([stats_brute(c) for c, _ in examples])

[[1.0, 3.0, 2.375, 2.5, 3.0], [1.0, 4.0, 2.1818181818181817, 2.0, 1.0]]


In [4]:
# Code nộp LeetCode cho Brute force (đúng nhưng chỉ dùng được khi tổng số phần tử nhỏ: n = 10^9 sẽ hết bộ nhớ)
class SolutionBrute:
    def sampleStats(self, count: List[int]) -> List[float]:
        sample = []
        for k in range(256):
            sample.extend([k] * count[k])
        n = len(sample)
        mean = sum(sample) / n
        median = sample[n // 2] if n % 2 else (sample[n // 2 - 1] + sample[n // 2]) / 2
        best, mode = -1, 0
        for k in range(256):
            if count[k] > best:
                best, mode = count[k], k
        return [float(sample[0]), float(sample[-1]), mean, float(median), float(mode)]

**Độ phức tạp:** `O(n)` thời gian và bộ nhớ với `n = Σ count` (có thể `10^9`: không khả thi).

**Nút thắt:** ta không cần từng phần tử, chỉ cần **tổng tích luỹ theo giá trị**.

## 6. Bước đột phá

Quét `k = 0..255`: cập nhật `n`, `S`, `min`, `max`, `mode` trong một lượt; tìm hai phần tử giữa bằng tổng tích luỹ ở lượt thứ hai (hoặc `bisect`). `O(256)` thời gian, `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** quét `256` ô + tìm phần tử thứ `k` (đáp án chính), **Cách 3** hai con trỏ từ hai đầu, **Cách 4** tổng tích luỹ + `bisect`, **Cách 5** số học chính xác bằng `Fraction`.

### Cách 2 — Quét `256` ô và tìm phần tử thứ `k` (đáp án chính)

**Giải thích:** lượt 1: `n`, `S`, `min`, `max`, `mode`. Lượt 2: `median = (x_{(n+1)//2} + x_{n//2+1}) / 2` bằng `kth_smallest`.

In [5]:
def stats_scan(count):
    n = S = 0
    lo, hi = None, None
    mode, best = 0, -1
    for v in range(256):
        c = count[v]
        if c:
            n += c
            S += v * c
            if lo is None:
                lo = v
            hi = v
            if c > best:
                best, mode = c, v
    median = (kth_smallest(count, (n + 1) // 2) + kth_smallest(count, n // 2 + 1)) / 2
    return [float(lo), float(hi), S / n, float(median), float(mode)]


for c, want in examples:
    assert close(stats_scan(c), want)
print([stats_scan(c) for c, _ in examples])

[[1.0, 3.0, 2.375, 2.5, 3.0], [1.0, 4.0, 2.1818181818181817, 2.0, 1.0]]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionScan:
    def sampleStats(self, count: List[int]) -> List[float]:
        n = total = 0
        lo = hi = None
        mode, best = 0, -1
        for v in range(256):
            c = count[v]
            if c:
                n += c
                total += v * c
                if lo is None:
                    lo = v
                hi = v
                if c > best:
                    best, mode = c, v

        def kth(j):
            run = 0
            for v in range(256):
                run += count[v]
                if run >= j:
                    return v

        median = (kth((n + 1) // 2) + kth(n // 2 + 1)) / 2
        return [float(lo), float(hi), total / n, float(median), float(mode)]

**Độ phức tạp:** `O(256)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Hai con trỏ từ hai đầu

**Ý tưởng:** đặt con trỏ `lo` ở giá trị nhỏ nhất, `hi` ở giá trị lớn nhất, mỗi bên có số lượng còn lại `cl`, `ch`. **Gỡ bỏ cùng lúc** từ hai đầu một số lượng bằng nhau, sao cho vẫn còn ít nhất `1` phần tử; còn lại `1` hoặc `2` phần tử là **phần tử giữa** (hoặc hai phần tử giữa). Bước gỡ theo khối `step = min(cl, ch, (rem - 1) // 2)` nên chỉ mất tối đa `~512` bước.

In [7]:
def median_two_pointers(count):
    lo = next(v for v in range(256) if count[v])
    hi = next(v for v in range(255, -1, -1) if count[v])
    rem = sum(count)
    cl, ch = count[lo], count[hi]
    while rem > 2:
        if lo == hi:
            return float(lo)                              # toàn bộ phần còn lại cùng một giá trị
        step = min(cl, ch, (rem - 1) // 2)
        cl -= step
        ch -= step
        rem -= 2 * step
        if cl == 0:
            lo += 1
            while count[lo] == 0:
                lo += 1
            cl = ch if lo == hi else count[lo]            # khi hai con trỏ gặp nhau chúng dùng chung một số đếm
        if ch == 0:
            hi -= 1
            while count[hi] == 0:
                hi -= 1
            ch = cl if lo == hi else count[hi]
    if lo == hi:
        return float(lo)                                  # còn 1 phần tử, hoặc 2 phần tử bằng nhau
    return (lo + hi) / 2                                  # còn đúng 2 phần tử: một ở lo, một ở hi


def stats_two_pointers(count):
    s = stats_scan(count)
    s[3] = median_two_pointers(count)
    return s


for c, want in examples:
    assert close(stats_two_pointers(c), want), stats_two_pointers(c)
print([stats_two_pointers(c) for c, _ in examples])

[[1.0, 3.0, 2.375, 2.5, 3.0], [1.0, 4.0, 2.1818181818181817, 2.0, 1.0]]


Sau vòng lặp còn `1` hoặc `2` phần tử. Nếu `lo < hi` thì `cl, ch ≥ 1` nên còn đúng `2` phần tử: một ở `lo`, một ở `hi`, trung vị là `(lo + hi) / 2`. Nếu `lo == hi` thì mọi phần tử còn lại bằng nhau, trung vị là `lo`. Ta sẽ kiểm chứng kỹ ở mục 12D (mọi mảng nhỏ).

In [8]:
# Code nộp LeetCode cho Cách 3 (chỉ phần tính trung vị khác; các thống kê còn lại như Cách 2)
class SolutionTwoPointers:
    def sampleStats(self, count: List[int]) -> List[float]:
        n = total = 0
        lo_v = hi_v = None
        mode, best = 0, -1
        for v in range(256):
            c = count[v]
            if c:
                n += c
                total += v * c
                if lo_v is None:
                    lo_v = v
                hi_v = v
                if c > best:
                    best, mode = c, v
        lo, hi = lo_v, hi_v
        rem = n
        cl, ch = count[lo], count[hi]
        median = None
        while rem > 2:
            if lo == hi:
                median = float(lo)
                break
            step = min(cl, ch, (rem - 1) // 2)
            cl -= step
            ch -= step
            rem -= 2 * step
            if cl == 0:
                lo += 1
                while count[lo] == 0:
                    lo += 1
                cl = ch if lo == hi else count[lo]
            if ch == 0:
                hi -= 1
                while count[hi] == 0:
                    hi -= 1
                ch = cl if lo == hi else count[hi]
        if median is None:
            median = float(lo) if lo == hi else (lo + hi) / 2
        return [float(lo_v), float(hi_v), total / n, median, float(mode)]

**Độ phức tạp:** `O(256)` thời gian, `O(1)` bộ nhớ.

### Cách 4 — Tổng tích luỹ + `bisect`

**Ý tưởng:** `pre = list(accumulate(count))` là mảng không giảm; `x_j` là chỉ số nhỏ nhất `v` với `pre[v] ≥ j`, tức `bisect_left(pre, j)`.

In [9]:
def stats_bisect(count):
    pre = list(accumulate(count))
    n = pre[-1]
    S = sum(v * c for v, c in enumerate(count))
    nz = [v for v in range(256) if count[v]]
    mode = max(range(256), key=count.__getitem__)
    x = lambda j: bisect_left(pre, j)                      # phần tử thứ j (1-based)
    median = (x((n + 1) // 2) + x(n // 2 + 1)) / 2
    return [float(nz[0]), float(nz[-1]), S / n, float(median), float(mode)]


for c, want in examples:
    assert close(stats_bisect(c), want)
print([stats_bisect(c) for c, _ in examples])

[[1.0, 3.0, 2.375, 2.5, 3.0], [1.0, 4.0, 2.1818181818181817, 2.0, 1.0]]


In [10]:
# Code nộp LeetCode cho Cách 4
class SolutionBisect:
    def sampleStats(self, count: List[int]) -> List[float]:
        pre = list(accumulate(count))
        n = pre[-1]
        total = sum(v * c for v, c in enumerate(count))
        nz = [v for v in range(256) if count[v]]
        mode = max(range(256), key=count.__getitem__)
        x = lambda j: bisect_left(pre, j)
        median = (x((n + 1) // 2) + x(n // 2 + 1)) / 2
        return [float(nz[0]), float(nz[-1]), total / n, float(median), float(mode)]

**Độ phức tạp:** `O(256)` dựng `pre`, mỗi truy vấn `O(log 256)`; `O(256)` bộ nhớ.

### Cách 5 — Số học chính xác bằng `Fraction`

**Ý tưởng:** tính `mean` và `median` bằng **phân số** `Fraction` (không làm tròn), chỉ đổi sang `float` ở bước cuối. Hữu ích để chứng minh kết quả khác đáp án chính xác **không quá một ULP**, và cho phép so sánh **đúng tuyệt đối** trong test.

In [11]:
def stats_exact(count):
    n = sum(count)
    S = sum(v * c for v, c in enumerate(count))
    nz = [v for v in range(256) if count[v]]
    mode = max(range(256), key=count.__getitem__)
    run, xs = 0, {}
    targets = {(n + 1) // 2, n // 2 + 1}
    for v in range(256):
        for j in list(targets):
            if j not in xs and run < j <= run + count[v]:
                xs[j] = v
        run += count[v]
    median = Fraction(xs[(n + 1) // 2] + xs[n // 2 + 1], 2)
    return [Fraction(nz[0]), Fraction(nz[-1]), Fraction(S, n), median, Fraction(mode)]


for c, want in examples:
    ex = stats_exact(c)
    assert close([float(v) for v in ex], want)
print([[str(v) for v in stats_exact(c)] for c, _ in examples])

[['1', '3', '19/8', '5/2', '3'], ['1', '4', '24/11', '2', '1']]


In [12]:
# Code nộp LeetCode cho Cách 5 (tính bằng Fraction, trả về float)
class SolutionExact:
    def sampleStats(self, count: List[int]) -> List[float]:
        n = sum(count)
        total = sum(v * c for v, c in enumerate(count))
        nz = [v for v in range(256) if count[v]]
        mode = max(range(256), key=count.__getitem__)
        xs, run = {}, 0
        targets = {(n + 1) // 2, n // 2 + 1}
        for v in range(256):
            for j in targets:
                if j not in xs and run < j <= run + count[v]:
                    xs[j] = v
            run += count[v]
        median = Fraction(xs[(n + 1) // 2] + xs[n // 2 + 1], 2)
        return [float(nz[0]), float(nz[-1]), float(Fraction(total, n)), float(median), float(mode)]

**Độ phức tạp:** `O(256)` thời gian; phân số có tử và mẫu tới `~10^11` nên chi phí hằng số nhỏ.

## 8. TRACE từng bước

Trace **Cách 2** với ví dụ 1: `count[1] = 1`, `count[2] = 3`, `count[3] = 4`.

| `v` | `count[v]` | `n` | `S` | `min/max` | `mode` (đếm lớn nhất) |
|---|---|---|---|---|---|
| 1 | 1 | 1 | 1 | min = 1 | 1 (đếm 1) |
| 2 | 3 | 4 | 7 | | 2 (đếm 3) |
| 3 | 4 | 8 | 19 | max = 3 | 3 (đếm 4) |

`mean = 19 / 8 = 2.375`. Trung vị: `n = 8` chẵn nên cần `x_4` và `x_5`. Tổng tích luỹ `1, 4, 8`: `x_4 = 2` (tích luỹ tại `v = 2` là `4 ≥ 4`), `x_5 = 3` (tại `v = 3` là `8 ≥ 5`) ⇒ `median = (2 + 3) / 2 = 2.5`. Kết quả `[1, 3, 2.375, 2.5, 3]`. Hàm trace in từng bước:

In [13]:
def trace_scan(count):
    n = S = 0
    run_info = []
    lo = hi = None
    mode, best = 0, -1
    for v in range(256):
        c = count[v]
        if c:
            n += c
            S += v * c
            lo = v if lo is None else lo
            hi = v
            if c > best:
                best, mode = c, v
            run_info.append((v, c, n))
            print(f"  v = {v}: count = {c}, n = {n}, S = {S}, tích luỹ = {n}, mode tạm thời = {mode}")
    a, b = (n + 1) // 2, n // 2 + 1
    xa, xb = kth_smallest(count, a), kth_smallest(count, b)
    print(f"  n = {n}: cần x_{a} = {xa} và x_{b} = {xb} -> median = {(xa + xb) / 2}")
    res = [float(lo), float(hi), S / n, (xa + xb) / 2, float(mode)]
    print("  kết quả:", res)
    return res


assert close(trace_scan(examples[0][0]), [1, 3, 2.375, 2.5, 3])
assert close(trace_scan(examples[1][0]), [1, 4, 24 / 11, 2, 1])

  v = 1: count = 1, n = 1, S = 1, tích luỹ = 1, mode tạm thời = 1
  v = 2: count = 3, n = 4, S = 7, tích luỹ = 4, mode tạm thời = 2
  v = 3: count = 4, n = 8, S = 19, tích luỹ = 8, mode tạm thời = 3
  n = 8: cần x_4 = 2 và x_5 = 3 -> median = 2.5
  kết quả: [1.0, 3.0, 2.375, 2.5, 3.0]
  v = 1: count = 4, n = 4, S = 4, tích luỹ = 4, mode tạm thời = 1
  v = 2: count = 3, n = 7, S = 10, tích luỹ = 7, mode tạm thời = 1
  v = 3: count = 2, n = 9, S = 16, tích luỹ = 9, mode tạm thời = 1
  v = 4: count = 2, n = 11, S = 24, tích luỹ = 11, mode tạm thời = 1
  n = 11: cần x_6 = 2 và x_6 = 2 -> median = 2.0
  kết quả: [1.0, 4.0, 2.1818181818181817, 2.0, 1.0]


**Trace ví dụ 2:** `n = 11` lẻ ⇒ `x_6` hai lần: tích luỹ `4, 7, 9, 11` ⇒ `x_6 = 2` (`7 ≥ 6`) ⇒ `median = 2`; `S = 4 + 6 + 6 + 8 = 24`, `mean = 24/11 = 2.18181…`; mode `1` (đếm `4`).

## 9. Code nộp LeetCode

Chọn **Cách 2** (quét `256` ô, tìm phần tử thứ `k` bằng tích luỹ). Giữ nguyên `class Solution`, `sampleStats(self, count)`:

In [14]:
class Solution:
    def sampleStats(self, count: List[int]) -> List[float]:
        n = total = 0                                   # số phần tử và tổng
        lo = hi = None
        mode, best = 0, -1
        for v in range(256):
            c = count[v]
            if c:
                n += c
                total += v * c
                if lo is None:
                    lo = v                              # giá trị nhỏ nhất có mặt
                hi = v                                  # giá trị lớn nhất có mặt
                if c > best:                            # mode duy nhất: ô đếm lớn nhất
                    best, mode = c, v

        def kth(j: int) -> int:                         # phần tử thứ j (1-based) khi mẫu đã sắp xếp
            run = 0
            for v in range(256):
                run += count[v]
                if run >= j:
                    return v

        median = (kth((n + 1) // 2) + kth(n // 2 + 1)) / 2     # n lẻ: hai số hạng bằng nhau
        return [float(lo), float(hi), total / n, float(median), float(mode)]


print([Solution().sampleStats(c) for c, _ in examples])
assert all(close(Solution().sampleStats(c), w) for c, w in examples)

[[1.0, 3.0, 2.375, 2.5, 3.0], [1.0, 4.0, 2.1818181818181817, 2.0, 1.0]]


## 10. Vì sao đúng

**(a) Mẫu đã sắp xếp.** Mẫu `x_1 ≤ … ≤ x_n` gồm `count[0]` số `0`, tiếp theo `count[1]` số `1`, … Ta có `x_j = v` ⇔ `Σ_{u<v} count[u] < j ≤ Σ_{u≤v} count[u]`; `kth(j)` trả về `v` nhỏ nhất có tổng tích luỹ `≥ j`, chính là giá trị đó (tính chất của hàm tích luỹ không giảm).

**(b) Trung vị.** Theo định nghĩa: `n` lẻ ⇒ `x_{(n+1)/2}`; `n` chẵn ⇒ `(x_{n/2} + x_{n/2+1}) / 2`. Với `n` lẻ: `(n+1)//2 = (n+1)/2 = n//2 + 1`, công thức chung cho `(x_{(n+1)/2} + x_{(n+1)/2}) / 2 = x_{(n+1)/2}`. Với `n` chẵn: `(n+1)//2 = n/2`, `n//2 + 1 = n/2 + 1`. Đúng.

**(c) Các thống kê còn lại.** `min`/`max` là giá trị nhỏ/lớn nhất có `count > 0` (vì giá trị không có mặt không nằm trong mẫu); `mean = Σ v·count[v] / n` là định nghĩa; `mode` là giá trị có `count` lớn nhất (duy nhất theo đề, nên so sánh `>` chặt là đủ).

**(d) Độ chính xác.** `total` và `n` là số nguyên (Python lớn tuỳ ý, thực tế `< 2^53`: `total ≤ 255·10^9 ≈ 2.6·10^11`), phép chia `total / n` được làm tròn đúng chuẩn IEEE: sai số tương đối `≤ 2^-53`, nhỏ hơn `10^-5` rất nhiều. Trung vị là `(a + b) / 2` với `a, b` nguyên `≤ 255`, chính xác.

**(e) Các cách còn lại.** Cách 3 loại bỏ đồng thời hai đầu nên phần còn lại luôn chứa các phần tử giữa; Cách 4 dùng `bisect_left` thay vòng lặp (cùng định nghĩa `x_j`); Cách 5 giữ phân số tới cuối; Cách 1 là định nghĩa.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp oracle dựng-lại-mẫu (và thư viện `statistics`) trên mọi mảng đếm nhỏ (mục 12D) và ngẫu nhiên, và khớp nhau trên các mảng có tổng tới `10^9` (mục 12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Dựng lại mẫu | `O(n)`, `n` tới `10^9` | `O(n)` | oracle |
| 2. Quét `256` ô | `O(256)` | `O(1)` | **đáp án chính** |
| 3. Hai con trỏ | `O(256)` | `O(1)` | không cần tích luỹ |
| 4. Tích luỹ + `bisect` | `O(256 + log 256)` | `O(256)` | gọn |
| 5. `Fraction` | `O(256)` | `O(1)` | chính xác tuyệt đối |

Đo thực tế (tổng `≈ 9.8·10^8`):

In [15]:
def bench(fn, arg, repeat=200):
    best = 1e9
    for _ in range(5):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(arg)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


big = [0] * 256
for v in range(256):
    big[v] = 3_800_000 + v * 100
big[100] += 50_000                                      # mode duy nhất
assert sum(big) <= 10 ** 9
for name, fn in [("scan", stats_scan), ("two_pointers", stats_two_pointers), ("bisect", stats_bisect), ("exact", stats_exact)]:
    print(f"{name:13s}: {bench(fn, big) * 1e6:8.1f} us -> {[round(float(x), 5) for x in fn(big)]}")

scan         :     53.0 us -> [0.0, 255.0, 127.64182, 128.0, 100.0]


two_pointers :    149.9 us -> [0.0, 255.0, 127.64182, 128.0, 100.0]
bisect       :     29.1 us -> [0.0, 255.0, 127.64182, 128.0, 100.0]


exact        :    130.7 us -> [0.0, 255.0, 127.64182, 128.0, 100.0]


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng đếm nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

Mọi so sánh số thực dùng `close` (sai số `10^-5`); riêng các cách tính bằng số học chính xác còn được so **đúng tuyệt đối** với `Fraction`.

### A. Ví dụ của đề

In [16]:
IMPLS = {
    "scan": stats_scan,
    "two_pointers": stats_two_pointers,
    "bisect": stats_bisect,
    "exact": lambda c: [float(v) for v in stats_exact(c)],
    "Solution": lambda c: Solution().sampleStats(c[:]),
    "SolutionScan": lambda c: SolutionScan().sampleStats(c[:]),
    "SolutionTwoPointers": lambda c: SolutionTwoPointers().sampleStats(c[:]),
    "SolutionBisect": lambda c: SolutionBisect().sampleStats(c[:]),
    "SolutionExact": lambda c: SolutionExact().sampleStats(c[:]),
}
BRUTE = {"brute": stats_brute, "SolutionBrute": lambda c: SolutionBrute().sampleStats(c[:])}


def statistics_oracle(count):
    """Oracle thứ hai dùng thư viện statistics trên mẫu dựng lại."""
    sample = [k for k in range(256) for _ in range(count[k])]
    modes = statistics.multimode(sample)
    assert len(modes) == 1                                # mode duy nhất
    return [float(min(sample)), float(max(sample)), float(statistics.mean(sample)), float(statistics.median(sample)), float(modes[0])]


def check(count, want=None, small=False):
    if want is None:
        want = statistics_oracle(count) if small else IMPLS["scan"](count)
    impls = dict(IMPLS)
    if small:
        impls.update(BRUTE)
    for name, fn in impls.items():
        got = fn(count)
        assert close(got, want), (name, {k: v for k, v in enumerate(count) if v}, got, want)
    return want


for c, want in examples:
    check(c, want, small=True)
print("ví dụ đề: mọi cài đặt đúng trong sai số 1e-5;", len(IMPLS), "cách nhanh +", len(BRUTE), "vét cạn")

ví dụ đề: mọi cài đặt đúng trong sai số 1e-5; 9 cách nhanh + 2 vét cạn


### B. Edge case

- Một phần tử duy nhất: `[v, v, v, v, v]`.
- Mọi phần tử bằng nhau (mode duy nhất).
- `min = 0` và `max = 255` (biên giá trị).
- Số phần tử **chẵn** có hai phần tử giữa khác nhau: trung vị là số thập phân `.5`.
- Số phần tử **lẻ**.
- Hai phần tử giữa **bằng nhau** dù `n` chẵn.
- Giá trị ở hai đầu cách xa nhau (`0` và `255`) nhưng mode ở giữa.
- Tổng số phần tử `10^9` (một ô có `10^9`).

In [17]:
edge = [
    (from_sample([7]), [7, 7, 7, 7, 7]),
    (from_sample([5, 5, 5]), [5, 5, 5, 5, 5]),
    (from_sample([0, 255, 255]), [0, 255, 170, 255, 255]),
    (from_sample([0, 0, 255]), [0, 255, 85, 0, 0]),
    (from_sample([1, 3, 3]), [1, 3, 7 / 3, 3, 3]),
    (from_sample([1, 2, 3, 3]), [1, 3, 9 / 4, 2.5, 3]),
    (from_sample([2, 2, 4, 4, 4]), [2, 4, 16 / 5, 4, 4]),
    (from_sample([10, 20, 20, 30, 30, 30]), [10, 30, 140 / 6, 25, 30]),
    (from_sample([0, 100, 100, 100, 255]), [0, 255, 111, 100, 100]),
]
big_one = [0] * 256
big_one[255] = 10 ** 9
edge.append((big_one, [255, 255, 255, 255, 255]))
for c, want in edge:
    got = check(c, want, small=(sum(c) <= 50))
    print(f"  n = {sum(c):>10}: {[round(x, 5) for x in got]}")
print(len(edge), "edge case: mọi cài đặt khớp")

  n =          1: [7, 7, 7, 7, 7]
  n =          3: [5, 5, 5, 5, 5]
  n =          3: [0, 255, 170, 255, 255]
  n =          3: [0, 255, 85, 0, 0]
  n =          3: [1, 3, 2.33333, 3, 3]
  n =          4: [1, 3, 2.25, 2.5, 3]
  n =          5: [2, 4, 3.2, 4, 4]
  n =          6: [10, 30, 23.33333, 25, 30]
  n =          5: [0, 255, 111, 100, 100]
  n = 1000000000: [255, 255, 255, 255, 255]
10 edge case: mọi cài đặt khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_integer_mean(count):                            # chia nguyên khi tính trung bình
    r = stats_scan(count)
    n, S = sum(count), sum(v * c for v, c in enumerate(count))
    r[2] = float(S // n)
    return r


def bug_lower_median(count):                            # n chẵn: chỉ lấy phần tử giữa dưới
    r = stats_scan(count)
    n = sum(count)
    r[3] = float(kth_smallest(count, (n + 1) // 2))
    return r


def bug_upper_median(count):                            # luôn lấy x_{n//2 + 1}
    r = stats_scan(count)
    n = sum(count)
    r[3] = float(kth_smallest(count, n // 2 + 1))
    return r


def bug_index_bounds(count):                            # min/max lấy 0 và 255 thay vì giá trị có mặt
    r = stats_scan(count)
    r[0], r[1] = 0.0, 255.0
    return r


def bug_unweighted_mean(count):                         # trung bình các giá trị có mặt, bỏ qua số đếm
    r = stats_scan(count)
    nz = [v for v in range(256) if count[v]]
    r[2] = sum(nz) / len(nz)
    return r


def bug_mode_by_value(count):                           # mode = giá trị lớn nhất có mặt (nhầm với max)
    r = stats_scan(count)
    r[4] = r[1]
    return r


def bug_median_unweighted(count):                       # trung vị của các GIÁ TRỊ có mặt, bỏ qua số đếm
    r = stats_scan(count)
    nz = [v for v in range(256) if count[v]]
    r[3] = float(statistics.median(nz))
    return r


def valid_counts(rng, support, max_count):
    while True:
        c = [0] * 256
        for p in support:
            c[p] = rng.randint(0, max_count)
        top = sorted(c, reverse=True)
        if sum(c) >= 1 and top[0] > top[1]:               # tổng >= 1 và mode duy nhất
            return c


BUGS = [bug_integer_mean, bug_lower_median, bug_upper_median, bug_index_bounds, bug_unweighted_mean, bug_mode_by_value, bug_median_unweighted]
rng = random.Random(10930)
pool = [valid_counts(rng, rng.sample(range(256), rng.randint(1, 6)), rng.choice([1, 3, 6])) for _ in range(800)]
for bug in BUGS:
    found = next((c for c in pool if not close(bug(c), statistics_oracle(c))), None)
    assert found, bug.__name__
    print(f"{bug.__name__:24s} sai trên {{{', '.join(f'{k}: {v}' for k, v in enumerate(found) if v)}}}: ra {[round(x, 4) for x in bug(found)]}, đúng {[round(x, 4) for x in statistics_oracle(found)]}")

bug_integer_mean         sai trên {228: 5, 239: 4}: ra [228.0, 239.0, 232.0, 228.0, 228.0], đúng [228.0, 239.0, 232.8889, 228.0, 228.0]
bug_lower_median         sai trên {49: 2, 138: 3, 179: 4, 248: 1}: ra [49.0, 248.0, 147.6, 138.0, 179.0], đúng [49.0, 248.0, 147.6, 158.5, 179.0]
bug_upper_median         sai trên {49: 2, 138: 3, 179: 4, 248: 1}: ra [49.0, 248.0, 147.6, 179.0, 179.0], đúng [49.0, 248.0, 147.6, 158.5, 179.0]
bug_index_bounds         sai trên {64: 1}: ra [0.0, 255.0, 64.0, 64.0, 64.0], đúng [64.0, 64.0, 64.0, 64.0, 64.0]
bug_unweighted_mean      sai trên {228: 5, 239: 4}: ra [228.0, 239.0, 233.5, 228.0, 228.0], đúng [228.0, 239.0, 232.8889, 228.0, 228.0]
bug_mode_by_value        sai trên {228: 5, 239: 4}: ra [228.0, 239.0, 232.8889, 228.0, 239.0], đúng [228.0, 239.0, 232.8889, 228.0, 228.0]
bug_median_unweighted    sai trên {228: 5, 239: 4}: ra [228.0, 239.0, 232.8889, 233.5, 228.0], đúng [228.0, 239.0, 232.8889, 228.0, 228.0]


### D. Vét cạn mọi mảng đếm nhỏ

Chọn `5` vị trí đại diện `(0, 1, 2, 254, 255)` và mọi bộ số đếm `c_i ∈ {0, 1, 2, 3}` (`4^5 = 1 024` mảng); giữ các mảng có tổng `≥ 1` và mode duy nhất. Mỗi mảng so với oracle dựng-lại-mẫu và với `statistics`. Làm lại với vị trí `(3, 4, 100, 101, 128)` để đa dạng khoảng cách.

In [19]:
t0 = time.perf_counter()
cnt = 0
for positions in [(0, 1, 2, 254, 255), (3, 4, 100, 101, 128)]:
    for counts in itertools.product(range(4), repeat=5):
        c = [0] * 256
        for p, k in zip(positions, counts):
            c[p] = k
        top = sorted(c, reverse=True)
        if sum(c) < 1 or top[0] == top[1]:
            continue
        want = statistics_oracle(c)
        assert close(stats_brute(c), want)
        for name, fn in IMPLS.items():
            assert close(fn(c), want), (name, positions, counts)
        cnt += 1
print(cnt, "mảng đếm hợp lệ:", len(IMPLS), "cài đặt khớp oracle và statistics;", round(time.perf_counter() - t0, 1), "s")

980 mảng đếm hợp lệ: 9 cài đặt khớp oracle và statistics; 0.3 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Mảng ngẫu nhiên nhỏ: mọi cách khớp `statistics`; (2) **nhân số đếm với `c`** (`c ≥ 1`) không đổi `min`, `max`, `mean`, `median`, `mode`; (3) `min ≤ median ≤ max`, `min ≤ mean ≤ max`; (4) giá trị `mode` có số đếm lớn nhất; (5) thêm một phần tử bằng **đúng trung vị** (khi `n` lẻ) không đổi trung vị; (6) hai cách chính xác (Cách 5) và Cách 2 cho **cùng một số thực** (cả hai làm tròn đúng).

In [20]:
rng = random.Random(20240193)
t0 = time.perf_counter()
for it in range(600):
    c = valid_counts(rng, rng.sample(range(256), rng.randint(1, 8)), rng.choice([1, 2, 5, 20]))
    want = check(c, small=True)                                                       # (1)
    k = rng.randint(2, 6)
    assert close(stats_scan([x * k for x in c]), want)                                # (2)
    assert want[0] <= want[3] <= want[1] and want[0] <= want[2] <= want[1]            # (3)
    assert c[int(want[4])] == max(c)                                                  # (4)
    n = sum(c)
    if n % 2 == 1:
        c2 = c[:]
        c2[int(want[3])] += 1
        if sorted(c2, reverse=True)[0] > sorted(c2, reverse=True)[1]:
            assert close(stats_scan(c2), [want[0], want[1], stats_scan(c2)[2], want[3], stats_scan(c2)[4]])   # (5)
    assert [float(v) for v in stats_exact(c)] == stats_scan(c)                       # (6)
print("(1)-(6) 600 mảng ngẫu nhiên:", round(time.perf_counter() - t0, 1), "s")

(1)-(6) 600 mảng ngẫu nhiên: 0.3 s


### F. Giới hạn

Tổng số phần tử tới `10^9`: một ô đầy, hai đầu lớn nhất, đều nhau (+ mode duy nhất), phân bố hình chuông, tăng dần. Chỉ so các cách nhanh với nhau và với số học chính xác `Fraction`, cùng các bất biến (không dựng mẫu).

In [21]:
t0 = time.perf_counter()
cases = {}
c = [0] * 256
c[0] = 10 ** 9
cases["một ô 0"] = c
c = [0] * 256
c[0], c[255] = 499_999_999, 500_000_000
cases["hai đầu"] = c
c = [3_900_000] * 256
c[77] += 1
cases["đều nhau"] = c
c = [int(9 * 10 ** 6 * (2.718281828 ** (-((v - 128) / 40) ** 2 / 2))) for v in range(256)]
c[128] += 1
cases["hình chuông"] = c
c = [v for v in range(256)]
cases["tăng dần"] = c
for name, c in cases.items():
    assert 1 <= sum(c) <= 10 ** 9 and sorted(c, reverse=True)[0] > sorted(c, reverse=True)[1]
    want = stats_scan(c)
    for iname, fn in IMPLS.items():
        assert close(fn(c), want), (iname, name)
    ex = stats_exact(c)
    assert [float(v) for v in ex] == want
    assert want[0] <= want[3] <= want[1] and c[int(want[4])] == max(c)
    print(f"  {name:12s}: n = {sum(c):>10}, kết quả {[round(x, 5) for x in want]}")
print("giới hạn:", round(time.perf_counter() - t0, 2), "s")

  một ô 0     : n = 1000000000, kết quả [0.0, 0.0, 0.0, 0.0, 0.0]
  hai đầu     : n =  999999999, kết quả [0.0, 255.0, 127.5, 255.0, 255.0]
  đều nhau    : n =  998400001, kết quả [0.0, 255.0, 127.5, 127.0, 77.0]
  hình chuông : n =  901145209, kết quả [0.0, 255.0, 127.99236, 128.0, 128.0]
  tăng dần    : n =      32640, kết quả [1.0, 255.0, 170.33333, 181.0, 255.0]
giới hạn: 0.0 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chia nguyên khi tính trung bình

In [22]:
c = examples[0][0]
print("bug:", bug_integer_mean(c)[2], "| đúng:", Solution().sampleStats(c)[2])
assert bug_integer_mean(c)[2] == 2.0 and abs(Solution().sampleStats(c)[2] - 2.375) < 1e-9

bug: 2.0 | đúng: 2.375


**Vì sao sai:** đề yêu cầu số thực. `19 // 8 = 2` mất phần thập phân. Dùng `/` (chia thật) trong Python 3.

### Lỗi 2: quên trường hợp `n` chẵn (trung vị)

In [23]:
c = examples[0][0]
print("bug dưới:", bug_lower_median(c)[3], "| bug trên:", bug_upper_median(c)[3], "| đúng:", Solution().sampleStats(c)[3])
assert bug_lower_median(c)[3] == 2.0 and bug_upper_median(c)[3] == 3.0 and Solution().sampleStats(c)[3] == 2.5

bug dưới: 2.0 | bug trên: 3.0 | đúng: 2.5


**Vì sao sai:** với `n = 8`, trung vị là trung bình của `x_4 = 2` và `x_5 = 3`, tức `2.5`; chỉ lấy một trong hai phần tử giữa cho `2` hoặc `3`.

### Lỗi 3: lấy `min/max` bằng chỉ số `0` và `255`

In [24]:
c = examples[0][0]
print("bug:", bug_index_bounds(c)[:2], "| đúng:", Solution().sampleStats(c)[:2])
assert bug_index_bounds(c)[:2] == [0.0, 255.0] and Solution().sampleStats(c)[:2] == [1.0, 3.0]

bug: [0.0, 255.0] | đúng: [1.0, 3.0]


**Vì sao sai:** `min` và `max` là giá trị **có mặt** trong mẫu (`count > 0`), không phải biên `0..255` của khoảng cho phép.

### Lỗi 4: trung bình/trung vị không có trọng số

In [25]:
c = from_sample([1, 1, 1, 1, 100])
print("trung bình bug:", bug_unweighted_mean(c)[2], "| đúng:", Solution().sampleStats(c)[2])
print("trung vị bug:", bug_median_unweighted(from_sample([1, 1, 1, 1, 100, 200, 200]))[3], "| đúng:", Solution().sampleStats(from_sample([1, 1, 1, 1, 100, 200, 200]))[3])
assert bug_unweighted_mean(c)[2] == 50.5 and abs(Solution().sampleStats(c)[2] - 20.8) < 1e-9

trung bình bug: 50.5 | đúng: 20.8
trung vị bug: 100.0 | đúng: 1.0


**Vì sao sai:** mỗi giá trị xuất hiện `count[v]` lần; trung bình và trung vị phải tính trên **mẫu có lặp**, không phải trên danh sách các giá trị khác nhau.

### Lỗi 5: nhầm `mode` với `max`

In [26]:
c = from_sample([1, 1, 1, 5])
print("bug:", bug_mode_by_value(c)[4], "| đúng:", Solution().sampleStats(c)[4])
assert bug_mode_by_value(c)[4] == 5.0 and Solution().sampleStats(c)[4] == 1.0

bug: 5.0 | đúng: 1.0


**Vì sao sai:** `mode` là giá trị xuất hiện **nhiều lần nhất** (ô `count` lớn nhất), không phải giá trị lớn nhất.

### Lỗi 6 (bonus): dựng lại mẫu khi tổng tới `10^9`

In [27]:
c = [0] * 256
c[200] = 10 ** 9
print("đáp án hằng số thời gian:", Solution().sampleStats(c))
print("dựng mẫu sẽ cần ~8 GB chỉ cho con trỏ của danh sách (không thực hiện)")
assert Solution().sampleStats(c) == [200.0] * 5

đáp án hằng số thời gian: [200.0, 200.0, 200.0, 200.0, 200.0]
dựng mẫu sẽ cần ~8 GB chỉ cho con trỏ của danh sách (không thực hiện)


**Vì sao sai:** danh sách `10^9` phần tử chiếm hàng chục GB (và `~10^9` thao tác). Bảng đếm cho phép tính trong `O(256)`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "thống kê mô tả của một mẫu rất lớn nhưng chỉ có ít giá trị rời rạc, cho dưới dạng bảng đếm / histogram" → **quét bảng đếm**, tìm phần tử thứ `k` bằng tổng tích luỹ.

**Mẹo:**

1. `median = (x_{(n+1)//2} + x_{n//2+1}) / 2` — một công thức cho cả `n` chẵn và lẻ.
2. `x_j` = giá trị nhỏ nhất có tích luỹ `≥ j` (`bisect_left` trên `accumulate`).
3. `mean` bằng `Σ v·count[v] / n`, nhớ chia thật.
4. `min/max`: giá trị có mặt (`count > 0`).
5. So sánh số thực với sai số: `abs(a - b) <= 1e-5`.
6. Oracle: dựng lại mẫu nhỏ và dùng `statistics` (`mean`, `median`, `multimode`).

**Họ bài cùng pattern:** Find Median from Data Stream (295), Kth Smallest Element in a Sorted Matrix (378), Top K Frequent Elements (347), Sort an Array (912), Median of Two Sorted Arrays (4), Count Sorting (counting sort).

## 15. Biến thể

1. **Phần tử thứ `k`** (phân vị bất kỳ) từ bảng đếm: `kth_smallest`.
2. **Phương sai / độ lệch chuẩn** của mẫu: `E[x²] − E[x]²` từ số đếm.
3. **Mode khi có hòa** (trả về tất cả mode hoặc mode nhỏ nhất).
4. **Khoảng tứ phân vị (IQR)** `Q3 − Q1` bằng `kth_smallest`.

Cài đặt và đối chiếu mẫu dựng lại:

In [28]:
def population_variance(count):
    n = sum(count)
    S = sum(v * c for v, c in enumerate(count))
    S2 = sum(v * v * c for v, c in enumerate(count))
    return Fraction(S2, n) - Fraction(S, n) ** 2


def all_modes(count):
    top = max(count)
    return [v for v in range(256) if count[v] == top]


def iqr(count):
    """Q1 = phần tử thứ ceil(n/4), Q3 = phần tử thứ ceil(3n/4) (định nghĩa 'nearest rank')."""
    n = sum(count)
    q1 = kth_smallest(count, -(-n // 4))
    q3 = kth_smallest(count, -(-3 * n // 4))
    return q3 - q1


rng = random.Random(15)
for _ in range(300):
    c = [0] * 256
    for p in rng.sample(range(256), rng.randint(1, 6)):
        c[p] = rng.randint(0, 5)
    if sum(c) == 0:
        continue
    sample = [k for k in range(256) for _ in range(c[k])]
    n = len(sample)
    assert abs(float(population_variance(c)) - statistics.pvariance(sample)) < 1e-9
    assert all_modes(c) == sorted(statistics.multimode(sample))
    for j in range(1, n + 1):
        assert kth_smallest(c, j) == sample[j - 1]
    assert iqr(c) == sample[-(-3 * n // 4) - 1] - sample[-(-n // 4) - 1]
print("phân vị thứ k / phương sai / mọi mode / IQR: khớp mẫu dựng lại")

phân vị thứ k / phương sai / mọi mode / IQR: khớp mẫu dựng lại


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng lại mẫu | định nghĩa | `O(n)` | `O(n)` | oracle (mẫu nhỏ) |
| 2. Quét 256 ô | tích luỹ + `kth` | `O(256)` | `O(1)` | **đáp án chính** |
| 3. Hai con trỏ | gỡ từ hai đầu | `O(256)` | `O(1)` | không cần tích luỹ |
| 4. `accumulate` + `bisect` | tra phần tử thứ `j` | `O(256)` | `O(256)` | nhiều truy vấn phân vị |
| 5. `Fraction` | chính xác tới cuối | `O(256)` | `O(1)` | cần độ chính xác tuyệt đối |

Mọi cách khớp oracle và `statistics` trên mọi mảng đếm nhỏ; khớp nhau trên mẫu `10^9` phần tử.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy (mẫu cho dưới dạng danh sách, dùng `from_sample`):

In [29]:
for sample in ([1, 1, 2], [4, 4, 4, 4], [0, 255, 255], [3, 1, 2, 2]):
    print(sample, "->", [round(x, 5) for x in Solution().sampleStats(from_sample(sample))])

[1, 1, 2] -> [1.0, 2.0, 1.33333, 1.0, 1.0]
[4, 4, 4, 4] -> [4.0, 4.0, 4.0, 4.0, 4.0]
[0, 255, 255] -> [0.0, 255.0, 170.0, 255.0, 255.0]
[3, 1, 2, 2] -> [1.0, 3.0, 2.0, 2.0, 2.0]


### Bài 2 (tính tay)

Mẫu `[2, 2, 5, 9, 9, 9]` (`count[2] = 2`, `count[5] = 1`, `count[9] = 3`). Tính `min`, `max`, `mean`, `median` (chú ý `n = 6` chẵn), `mode`. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện tìm phần tử thứ `j` và trung vị:

In [30]:
def median_todo(count):
    n = sum(count)

    def kth(j):
        run = 0
        for v in range(256):
            pass        # TODO 1: run += count[v]; nếu run >= j thì trả về v
        return -1

    return 0.0          # TODO 2: (kth((n + 1) // 2) + kth(n // 2 + 1)) / 2


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(median_todo(c) == w[3] for c, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **`accumulate` + `bisect_left`**:

In [31]:
class MySolution:
    def sampleStats(self, count):
        pass            # TODO


# Khi làm xong:
# assert all(close(MySolution().sampleStats(c), w) for c, w in examples)

### Bài 5 (mini challenge): phân vị bất kỳ

Viết `percentile(count, p)` trả về phần tử ở **phân vị `p`** theo định nghĩa "nearest rank": `x_{ceil(p·n/100)}` với `1 ≤ p ≤ 100` (`p = 100` là `max`, `p = 50` trả về phần tử thứ `ceil(n/2)`). Kiểm tra: mẫu `[1,2,2,2,3,3,3,3]`: `percentile(·, 25) = 2` (`x_2`), `percentile(·, 100) = 3`, `percentile(·, 10) = 1`.

In [32]:
def percentile(count, p):
    pass            # TODO


# Khi làm xong:
# c = from_sample([1, 2, 2, 2, 3, 3, 3, 3])
# assert percentile(c, 25) == 2 and percentile(c, 100) == 3 and percentile(c, 10) == 1

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đếm `n`, tính tổng, tìm min/max/mode; trung vị theo `n` chẵn/lẻ.
- Mức 2: `[1,1,2]`: `n = 3`, trung vị `x_2 = 1`; `[0,255,255]`: trung vị `x_2 = 255`.
- Mức 3: `[1,2,1.33333,1,1]`, `[4,4,4,4,4]`, `[0,255,170,255,255]`, `[1,3,2,2,2]`.

**Bài 2**

- Mức 1: `n = 6`, `S = 4 + 5 + 27 = 36`.
- Mức 2: tích luỹ `2, 3, 6`; cần `x_3` và `x_4`: `x_3 = 5`, `x_4 = 9`.
- Mức 3: `[2, 9, 6, 7, 9]`.

**Bài 3**

- Mức 1: `run += count[v]`.
- Mức 2: `if run >= j: return v`.
- Mức 3: `return (kth((n + 1) // 2) + kth(n // 2 + 1)) / 2`.

**Bài 4**

- Mức 1: `pre = list(accumulate(count))`, `n = pre[-1]`.
- Mức 2: `bisect_left(pre, j)` cho `x_j`.
- Mức 3: `median = (x((n+1)//2) + x(n//2+1)) / 2`; `mode = max(range(256), key=count.__getitem__)`.

**Bài 5**

- Mức 1: `n = sum(count)`; chỉ số xếp hạng `r = ceil(p·n/100)`.
- Mức 2: `r = -(-p * n // 100)` (chia lên bằng số nguyên).
- Mức 3: `kth_smallest(count, r)`; đảm bảo `r ≥ 1`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
outs = [Solution().sampleStats(from_sample(s)) for s in ([1, 1, 2], [4, 4, 4, 4], [0, 255, 255], [3, 1, 2, 2])]
want = [[1, 2, 4 / 3, 1, 1], [4, 4, 4, 4, 4], [0, 255, 170, 255, 255], [1, 3, 2, 2, 2]]
print(outs)
assert all(close(o, w) for o, w in zip(outs, want))
print("Bài 1 OK")

[[1.0, 2.0, 1.3333333333333333, 1.0, 1.0], [4.0, 4.0, 4.0, 4.0, 4.0], [0.0, 255.0, 170.0, 255.0, 255.0], [1.0, 3.0, 2.0, 2.0, 2.0]]
Bài 1 OK


### Bài 2

In [34]:
c = from_sample([2, 2, 5, 9, 9, 9])
r = Solution().sampleStats(c)
print(r)
assert close(r, [2, 9, 6, 7, 9]) and close(statistics_oracle(c), r)
print("Bài 2 OK")

[2.0, 9.0, 6.0, 7.0, 9.0]
Bài 2 OK


### Bài 3

In [35]:
def median_answer(count):
    n = sum(count)

    def kth(j):
        run = 0
        for v in range(256):
            run += count[v]
            if run >= j:
                return v

    return (kth((n + 1) // 2) + kth(n // 2 + 1)) / 2


assert all(median_answer(c) == w[3] for c, w in examples)
for c in pool:
    assert median_answer(c) == statistics_oracle(c)[3]
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def sampleStats(self, count):
        pre = list(accumulate(count))
        n = pre[-1]
        S = sum(v * c for v, c in enumerate(count))
        nz = [v for v in range(256) if count[v]]
        x = lambda j: bisect_left(pre, j)
        return [float(nz[0]), float(nz[-1]), S / n, (x((n + 1) // 2) + x(n // 2 + 1)) / 2, float(max(range(256), key=count.__getitem__))]


assert all(close(MySolutionAnswer().sampleStats(c), w) for c, w in examples)
for c in pool:
    assert close(MySolutionAnswer().sampleStats(c), statistics_oracle(c))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def percentile(count, p):
    n = sum(count)
    r = max(1, -(-p * n // 100))
    return kth_smallest(count, r)


c = from_sample([1, 2, 2, 2, 3, 3, 3, 3])
assert percentile(c, 25) == 2 and percentile(c, 100) == 3 and percentile(c, 10) == 1
for cc in pool[:200]:
    sample = [k for k in range(256) for _ in range(cc[k])]
    n = len(sample)
    for p in (1, 10, 25, 50, 75, 90, 100):
        assert percentile(cc, p) == sample[max(1, -(-p * n // 100)) - 1]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (thống kê từ bảng đếm, không dựng mẫu):**

```python
class Solution:
    def sampleStats(self, count: List[int]) -> List[float]:
        n = total = 0
        lo = hi = None
        mode, best = 0, -1
        for v in range(256):
            c = count[v]
            if c:
                n += c
                total += v * c
                lo = v if lo is None else lo
                hi = v
                if c > best:
                    best, mode = c, v

        def kth(j):
            run = 0
            for v in range(256):
                run += count[v]
                if run >= j:
                    return v

        median = (kth((n + 1) // 2) + kth(n // 2 + 1)) / 2
        return [float(lo), float(hi), total / n, float(median), float(mode)]
```

**Checklist trước khi submit:**

- [ ] `min`/`max` lấy từ giá trị có `count > 0`, không phải `0`/`255`?
- [ ] `mean = total / n` bằng chia thật, `total` có trọng số `v * count[v]`?
- [ ] Trung vị dùng `(kth((n+1)//2) + kth(n//2+1)) / 2` cho cả `n` chẵn và lẻ?
- [ ] `mode` chọn theo **số đếm** lớn nhất?
- [ ] Không dựng lại mẫu (`n` tới `10^9`)?
- [ ] Đã thử `n` chẵn/lẻ, một phần tử, `min = 0`, `max = 255`, một ô `10^9`?
- [ ] Đúng tên `class Solution`, `sampleStats(self, count)`, trả về `float`, không in debug.